# Test Environment for Generative AI classroom labs

This lab provides a test environment for the codes generated using the Generative AI classroom.

Follow the instructions below to set up this environment for further use.


# Setup


### Install required libraries

In case of a requirement of installing certain python libraries for use in your task, you may do so as shown below.


In [1]:
%pip install seaborn
import piplite

await piplite.install(['nbformat', 'plotly'])

### Dataset URL from the GenAI lab
Use the URL provided in the GenAI lab in the cell below. 


In [3]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/laptop_pricing_dataset_mod1.csv"


### Downloading the dataset

Execute the following code to download the dataset in to the interface.

> Please note that this step is essential in JupyterLite. If you are using a downloaded version of this notebook and running it on JupyterLabs, then you can skip this step and directly use the URL in pandas.read_csv() function to read the dataset as a dataframe


In [4]:
from pyodide.http import pyfetch

async def download(url, filename):
    response = await pyfetch(url)
    if response.status == 200:
        with open(filename, "wb") as f:
            f.write(await response.bytes())

path = URL

await download(path, "dataset.csv")
file_name  = "dataset.csv"

---


# Test Environment


In [9]:
import pandas as pd

df = pd.read_csv("dataset.csv")

print(df.head())
print("Rows:", len(df), "Columns:", len(df.columns))

   Unnamed: 0 Manufacturer  Category     Screen  GPU  OS  CPU_core  \
0           0         Acer         4  IPS Panel    2   1         5   
1           1         Dell         3    Full HD    1   1         3   
2           2         Dell         3    Full HD    1   1         7   
3           3         Dell         4  IPS Panel    2   1         5   
4           4           HP         4    Full HD    2   1         7   

   Screen_Size_cm  CPU_frequency  RAM_GB  Storage_GB_SSD  Weight_kg  Price  
0          35.560            1.6       8             256       1.60    978  
1          39.624            2.0       4             256       2.20    634  
2          39.624            2.7       8             256       2.20    946  
3          33.782            1.6       8             128       1.22   1244  
4          39.624            1.8       8             256       1.91    837  
Rows: 238 Columns: 13


In [10]:
missing_counts = df.isnull().sum()
cols_with_missing = missing_counts[missing_counts > 0].index.tolist()
print("Columns with missing values:", cols_with_missing)
print("Missing value counts per column:\n", missing_counts)

Columns with missing values: ['Screen_Size_cm', 'Weight_kg']
Missing value counts per column:
 Unnamed: 0        0
Manufacturer      0
Category          0
Screen            0
GPU               0
OS                0
CPU_core          0
Screen_Size_cm    4
CPU_frequency     0
RAM_GB            0
Storage_GB_SSD    0
Weight_kg         5
Price             0
dtype: int64


In [11]:
# Replace missing Screen_Size_cm with mode
df['Screen_Size_cm'] = df['Screen_Size_cm'].fillna(df['Screen_Size_cm'].mode().iloc[0])
# Replace missing Weight_kg with mean
df['Weight_kg'] = df['Weight_kg'].fillna(df['Weight_kg'].mean())


In [12]:
print(df[['Screen_Size_cm','Weight_kg']].head())
print("Missing after imputation:\n", df[['Screen_Size_cm','Weight_kg']].isnull().sum())

   Screen_Size_cm  Weight_kg
0          35.560       1.60
1          39.624       2.20
2          39.624       2.20
3          33.782       1.22
4          39.624       1.91
Missing after imputation:
 Screen_Size_cm    0
Weight_kg         0
dtype: int64


In [14]:
df['Screen_Size_cm'] = df['Screen_Size_cm'].astype(float)
df['Weight_kg'] = df['Weight_kg'].astype(float)


In [15]:
df['Screen_Size_inch'] = df['Screen_Size_cm'] / 2.54
df['Weight_pounds'] = df['Weight_kg'] * 2.20462262185
df = df.drop(columns=['Screen_Size_cm', 'Weight_kg'])

In [16]:
# Create indicator variables for Screen as df1
df1 = pd.get_dummies(df['Screen'], prefix='Screen')
# Append indicators to df
df = pd.concat([df, df1], axis=1)
# Drop the original Screen column
df = df.drop(columns=['Screen'])


In [18]:
# Example: rate provided numerically
usd_to_eur = 0.92
df['Price_EUR'] = (df['Price'] * usd_to_eur).astype(float)

# Optional: fetch rate from an API (no error handling shown)
# import requests
# usd_to_eur = float(requests.get("https://api.exchangerate-api.com/v4/latest/USD").json()["rates"]["EUR"])
# df['Price_EUR'] = (df['Price'] * usd_to_eur).astype(float)


In [19]:
print(df[['Price', 'Price_EUR']].head())

   Price  Price_EUR
0    978     899.76
1    634     583.28
2    946     870.32
3   1244    1144.48
4    837     770.04


## Authors


[Abhishek Gagneja](https://www.linkedin.com/in/abhishek-gagneja-23051987/)


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2023-12-10|0.1|Abhishek Gagneja|Initial Draft created|


Copyright © 2023 IBM Corporation. All rights reserved.
